# Data Wrangling and Transformation (ETL)

After collecting raw data from multiple sources, the next critical step is to prepare the data for analysis and modeling. Raw datasets often contain inconsistencies such as missing values, noisy text, inconsistent formatting, and mixed data types.

The purpose of this notebook is to perform data wrangling, also referred to as the ETL (Extract, Transform, Load) process. This involves transforming raw data into a clean, structured format that is suitable for analysis and predictive modeling.

Key objectives in this stage include:

- Standardizing column names for consistency
- Cleaning textual data using regular expressions
- Extracting numerical values from mixed-format fields
- Handling missing values
- Encoding categorical variables into numerical format
- Normalizing numerical features for improved model performance

This step ensures that downstream processes such as exploratory data analysis and modeling operate on reliable and well-structured data.

## Import Required Libraries

The following libraries are used for data wrangling:

- `pandas`: for data manipulation and transformation
- `numpy`: for numerical operations
- `re`: for regular expression operations (text cleaning)

In [1]:
# Import libraries required for data wrangling

import pandas as pd        # Data manipulation and DataFrame operations
import numpy as np         # Numerical operations
import re                  # Regular expressions for text processing

## Load Raw Dataset

In this step, we load the raw dataset collected during the data collection phase. This dataset contains weather information and bike-sharing demand variables.

The data at this stage may contain inconsistencies such as missing values, mixed data types, and irregular formatting.

In [2]:
# Load dataset (update path as needed)

df = pd.read_csv("raw_seoul_bike_sharing.csv")

# Display first few rows
df.head()

,DATE,RENTED_BIKE_COUNT,HOUR,TEMPERATURE,HUMIDITY,WIND_SPEED,VISIBILITY,DEW_POINT_TEMPERATURE,SOLAR_RADIATION,RAINFALL,SNOWFALL,SEASONS,HOLIDAY,FUNCTIONING_DAY
0,01/12/2017,254.0,0,-5.2,37,2.2,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
1,01/12/2017,204.0,1,-5.5,38,0.8,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
2,01/12/2017,173.0,2,-6.0,39,1.0,2000,-17.7,0.0,0.0,0.0,Winter,No Holiday,Yes
3,01/12/2017,107.0,3,-6.2,40,0.9,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
4,01/12/2017,78.0,4,-6.0,36,2.3,2000,-18.6,0.0,0.0,0.0,Winter,No Holiday,Yes


## Standardize Column Names

Column names are standardized to ensure consistency and ease of use. This includes:

- Converting names to lowercase
- Replacing spaces with underscores
- Removing special characters

Consistent naming conventions simplify downstream data manipulation and modeling.

In [3]:
# Convert column names to lowercase
df.columns = df.columns.str.lower()

# Replace spaces with underscores
df.columns = df.columns.str.replace(" ", "_")

# Remove special characters if necessary
df.columns = df.columns.str.replace(r"[^\w\s]", "", regex=True)

# Display updated column names
df.columns

Index(['date', 'rented_bike_count', 'hour', 'temperature', 'humidity',
       'wind_speed', 'visibility', 'dew_point_temperature', 'solar_radiation',
       'rainfall', 'snowfall', 'seasons', 'holiday', 'functioning_day'],
      dtype='object')

## Clean Text Fields Using Regular Expressions

Some columns may contain unwanted characters such as units, symbols, or embedded text. Regular expressions are used to extract meaningful numerical values from these fields.

This step ensures that columns intended for numerical analysis are properly formatted.

In [4]:
# Example: Extract numeric values from a column containing mixed text

# Replace non-numeric characters
df['temperature'] = df['temperature'].astype(str).apply(
    lambda x: re.sub(r"[^0-9.\-]", "", x)
)

# Convert cleaned column to numeric type
df['temperature'] = pd.to_numeric(df['temperature'], errors='coerce')

## Handle Missing Values

Missing values can negatively impact analysis and model performance. We address missing data using the following strategies:

- Removing rows with excessive missing values
- Imputing missing numerical values with mean or median

In [5]:
# Check missing values
df.isnull().sum()

# Fill missing numerical values with column mean
df = df.fillna(df.mean(numeric_only=True))

# Verify missing values are handled
df.isnull().sum()

date                     0
rented_bike_count        0
hour                     0
temperature              0
humidity                 0
wind_speed               0
visibility               0
dew_point_temperature    0
solar_radiation          0
rainfall                 0
snowfall                 0
seasons                  0
holiday                  0
functioning_day          0
dtype: int64

## Create Dummy Variables for Categorical Features

Machine learning models require numerical input. Categorical variables such as seasons or holidays must be converted into numerical representations using one-hot encoding.

This process creates binary indicator variables for each category.

In [6]:
# Convert categorical columns into dummy variables

df = pd.get_dummies(df, columns=['seasons', 'holiday'], drop_first=True)

# Display updated dataset
df.head()

,date,rented_bike_count,hour,temperature,humidity,wind_speed,visibility,dew_point_temperature,solar_radiation,rainfall,snowfall,functioning_day,seasons_Spring,seasons_Summer,seasons_Winter,holiday_No Holiday
0,01/12/2017,254.0,0,-5.2,37,2.2,2000,-17.6,0.0,0.0,0.0,Yes,False,False,True,True
1,01/12/2017,204.0,1,-5.5,38,0.8,2000,-17.6,0.0,0.0,0.0,Yes,False,False,True,True
2,01/12/2017,173.0,2,-6.0,39,1.0,2000,-17.7,0.0,0.0,0.0,Yes,False,False,True,True
3,01/12/2017,107.0,3,-6.2,40,0.9,2000,-17.6,0.0,0.0,0.0,Yes,False,False,True,True
4,01/12/2017,78.0,4,-6.0,36,2.3,2000,-18.6,0.0,0.0,0.0,Yes,False,False,True,True


## Save Cleaned (Pre-Normalization) Dataset

We persist the cleaned-but-not-yet-normalized DataFrame as `cleaned_bike_sharing.csv`. Downstream notebooks (04 baseline, 05 refinement, 06 evaluation, 07 importance, 08 selection) load this snapshot — having it on disk before scaling means modeling notebooks can choose their own feature scaling without depending on the order of operations here.

In [ ]:
# ── Persist Cleaned (Pre-Normalization) Snapshot ─────────────
df.to_csv("cleaned_bike_sharing.csv", index=False)   # write cleaned-but-not-yet-scaled DataFrame for downstream modeling notebooks


## Normalize Numerical Features

Normalization ensures that numerical variables are on a comparable scale. This is particularly important for regression and regularization models.

We apply min-max scaling to bring all values into the range [0, 1].

In [8]:
# Select numerical columns
numeric_cols = df.select_dtypes(include=np.number).columns

# Apply min-max normalization
df[numeric_cols] = (df[numeric_cols] - df[numeric_cols].min()) / (
    df[numeric_cols].max() - df[numeric_cols].min()
)

# Display normalized data
df.head()

,date,rented_bike_count,hour,temperature,humidity,wind_speed,visibility,dew_point_temperature,solar_radiation,rainfall,snowfall,functioning_day,seasons_Spring,seasons_Summer,seasons_Winter,holiday_No Holiday
0,01/12/2017,0.070906,0.000000,0.220280,0.377551,0.297297,1.0,0.224913,0.0,0.0,0.0,Yes,False,False,True,True
1,01/12/2017,0.056837,0.043478,0.215035,0.387755,0.108108,1.0,0.224913,0.0,0.0,0.0,Yes,False,False,True,True
2,01/12/2017,0.048115,0.086957,0.206294,0.397959,0.135135,1.0,0.223183,0.0,0.0,0.0,Yes,False,False,True,True
3,01/12/2017,0.029544,0.130435,0.202797,0.408163,0.121622,1.0,0.224913,0.0,0.0,0.0,Yes,False,False,True,True
4,01/12/2017,0.021384,0.173913,0.206294,0.367347,0.310811,1.0,0.207612,0.0,0.0,0.0,Yes,False,False,True,True


## Save Normalized Dataset

We persist the fully normalized DataFrame as `seoul_bike_sharing_converted_normalized.csv` — the canonical input for any modeling step that expects features on the [0, 1] range.

In [ ]:
# ── Persist Normalized Dataset ────────────────────────────────
df.to_csv("seoul_bike_sharing_converted_normalized.csv", index=False)   # write normalized DataFrame as the canonical scaled input


## Summary

In this notebook, we transformed raw collected data into a clean and structured format by:

- Standardizing column names
- Cleaning and extracting numeric values using regular expressions
- Handling missing values
- Encoding categorical variables into numerical format
- Normalizing numerical features

This prepared dataset is now suitable for exploratory data analysis and predictive modeling in subsequent stages of the project.

---

## Author & Acknowledgment

**Author:**  
Deepan Mehta  

**GitHub Profile:**  
https://github.com/deepan-mehta-analytics

This notebook focuses on the data wrangling stage of the project, where raw datasets are transformed into a structured format suitable for analysis and modeling.

The concepts and workflow are inspired by instructional materials from the IBM Skills Network capstone labs, particularly those related to data cleaning using regular expressions and data transformation using structured data manipulation techniques.

Special acknowledgment is given to the original contributors of the lab materials, including:

- Yan Luo  
- Jeff Grossman  
- Rav Ahuja et al. at the IBM Skills Network  

These materials provided foundational guidance on handling missing values, feature engineering, and preparing datasets for machine learning workflows.

---

## Project Context

This notebook is part of a larger end-to-end data science pipeline, including:

- Data Collection (Web scraping, APIs, datasets)
- Data Wrangling and Transformation (ETL)
- Exploratory Data Analysis (EDA)
- Predictive Modeling (Regression & Regularization)
- Interactive Dashboard Development (R Shiny)

---

## Notes

All code and explanations in this notebook have been independently rewritten and enhanced to reflect a production-oriented workflow, ensuring clarity, reproducibility, and applicability to real-world data science problems.

The repository link will be updated with the finalized project structure, including notebooks, reports, and deployment components.

---